In [1]:
from google.colab import files

uploaded = files.upload()

Saving Q10_bank_fraud.csv to Q10_bank_fraud.csv


In [2]:
import pandas as pd
from scipy.stats import chi2_contingency

In [3]:
df = pd.read_csv("Q10_bank_fraud.csv")
print(df.head())

  transaction_id         timestamp  transaction_amount_inr time_of_day  \
0        TX00001  2026-03-02 08:09                 3258.54     Morning   
1        TX00002  2026-07-04 03:24                 1720.66         NaN   
2        TX00003  2026-04-28 00:31                 1182.08       Night   
3        TX00004  2026-04-11 04:46                 3416.84       Night   
4        TX00005  2026-05-30 15:57                 3014.57   Afternoon   

  fraud_status  
0   Legitimate  
1   Legitimate  
2   Legitimate  
3   Legitimate  
4   Legitimate  


In [4]:
print(df.shape)
print(df.columns)

(123, 5)
Index(['transaction_id', 'timestamp', 'transaction_amount_inr', 'time_of_day',
       'fraud_status'],
      dtype='object')


In [5]:
df = df.drop_duplicates(subset='transaction_id')

print("Records after removing duplicates:", len(df))

Records after removing duplicates: 120


In [6]:
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    errors='coerce'
)

print(df['timestamp'].head())

0   2026-03-02 08:09:00
1   2026-07-04 03:24:00
2   2026-04-28 00:31:00
3   2026-04-11 04:46:00
4   2026-05-30 15:57:00
Name: timestamp, dtype: datetime64[ns]


In [7]:
df = df.dropna(subset=['timestamp'])

print("Records after timestamp cleaning:", len(df))

Records after timestamp cleaning: 119


In [8]:
def category(h):
    if 5 <= h < 12:
        return "Morning"
    elif 12 <= h < 17:
        return "Afternoon"
    elif 17 <= h < 21:
        return "Evening"
    else:
        return "Night"

df['time_of_day'] = df['timestamp'].dt.hour.apply(category)

print(df[['timestamp', 'time_of_day']].head())

            timestamp time_of_day
0 2026-03-02 08:09:00     Morning
1 2026-07-04 03:24:00       Night
2 2026-04-28 00:31:00       Night
3 2026-04-11 04:46:00       Night
4 2026-05-30 15:57:00   Afternoon


In [9]:
print(df['transaction_amount_inr'].describe())

count      118.000000
mean     10145.787203
std      12838.934723
min        950.520000
25%       3143.520000
50%       6231.255000
75%      12268.855000
max      87857.280000
Name: transaction_amount_inr, dtype: float64


In [10]:
print(
    df.groupby('time_of_day')['transaction_amount_inr'].mean()
)

time_of_day
Afternoon    13983.772692
Evening      13038.360000
Morning       7818.313684
Night         8849.796000
Name: transaction_amount_inr, dtype: float64


In [11]:
table = pd.crosstab(
    df['time_of_day'],
    df['fraud_status']
)

print(table)

fraud_status  Fraud  Legitimate
time_of_day                    
Afternoon         1          25
Evening           1          13
Morning           5          33
Night             6          35


In [12]:
chi2, p, dof, expected = chi2_contingency(table)

print("Chi-Square:", chi2)
print("P-value:", p)

Chi-Square: 2.3190493432965917
P-value: 0.5088813847236417


In [13]:
if p < 0.05:
    print("There is a significant relationship.")
else:
    print("There is no significant relationship.")

There is no significant relationship.


In [14]:
df.to_csv("Q10_bank_fraud_cleaned.csv", index=False)

In [15]:
from google.colab import files

files.download("Q10_bank_fraud_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>